# Lesson 11B: Model-Based Reinforcement Learning - Practical

## Introduction

In Lesson 11A we covered the theory: learn a model of the environment, then plan with it. Here we build the pieces end to end:

1. A **forward model** learned by supervised regression on collected transitions
2. **Dyna-Q**: integrate real learning with planning against a learned tabular model
3. **Model Predictive Control (MPC)**: plan action sequences against the learned forward model
4. **MCTS**: tree search with UCB1 using the true simulator
5. A **sketch** of AlphaZero-style search: MCTS guided by a policy/value network

Environments: GridWorld (tabular Dyna-Q) and CartPole (forward model, MPC, MCTS).

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt
import copy
import math
from collections import defaultdict

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Part 1: Learning a Forward Model

A forward model predicts $p(s' \mid s, a)$. We treat it as a supervised learning problem: collect $(s, a, s', r)$ transitions from random rollouts, then regress a network onto the **state delta** $s' - s$ and the reward.

In [ ]:
class ForwardModel(nn.Module):
    """Predicts (next_state - state) and reward given (state, one-hot action)."""

    def __init__(self, state_dim, n_actions, hidden=64):
        super().__init__()
        self.n_actions = n_actions
        self.trunk = nn.Sequential(
            nn.Linear(state_dim + n_actions, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
        )
        self.delta_head = nn.Linear(hidden, state_dim)
        self.reward_head = nn.Linear(hidden, 1)

    def forward(self, state, action_onehot):
        h = self.trunk(torch.cat([state, action_onehot], dim=-1))
        return self.delta_head(h), self.reward_head(h)

    def predict(self, state, action):
        """Single-transition prediction. state: np.array, action: int. Returns (next_state, reward)."""
        with torch.no_grad():
            s = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            a = torch.zeros(1, self.n_actions, device=device)
            a[0, action] = 1.0
            delta, reward = self.forward(s, a)
        return (s.squeeze(0) + delta.squeeze(0)).cpu().numpy(), reward.item()


def collect_random_transitions(env, n_steps):
    transitions = []
    state, _ = env.reset(seed=0)
    for _ in range(n_steps):
        action = env.action_space.sample()
        next_state, reward, terminated, truncated, _ = env.step(action)
        transitions.append((state, action, next_state, reward))
        state, _ = env.reset() if (terminated or truncated) else (next_state, None)
    return transitions


def train_forward_model(model, transitions, n_actions, epochs=50, batch_size=64, lr=1e-3):
    states = torch.as_tensor(np.array([t[0] for t in transitions]), dtype=torch.float32, device=device)
    actions = torch.as_tensor([t[1] for t in transitions], dtype=torch.long, device=device)
    next_states = torch.as_tensor(np.array([t[2] for t in transitions]), dtype=torch.float32, device=device)
    rewards = torch.as_tensor([t[3] for t in transitions], dtype=torch.float32, device=device).unsqueeze(-1)
    actions_onehot = torch.nn.functional.one_hot(actions, n_actions).float()
    targets_delta = next_states - states

    optimizer = optim.Adam(model.parameters(), lr=lr)
    n = states.shape[0]
    losses = []
    for epoch in range(epochs):
        perm = torch.randperm(n)
        epoch_loss = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            pred_delta, pred_reward = model(states[idx], actions_onehot[idx])
            loss = nn.functional.mse_loss(pred_delta, targets_delta[idx]) + \
                   nn.functional.mse_loss(pred_reward, rewards[idx])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * idx.shape[0]
        losses.append(epoch_loss / n)
    return losses


cartpole_env = gym.make('CartPole-v1')
state_dim = cartpole_env.observation_space.shape[0]
n_actions = cartpole_env.action_space.n

forward_model = ForwardModel(state_dim, n_actions).to(device)
transitions = collect_random_transitions(cartpole_env, n_steps=4000)
losses = train_forward_model(forward_model, transitions, n_actions, epochs=30)
print(f"Forward model final training loss: {losses[-1]:.5f}")

In [ ]:
# Sanity check: compare model rollout to real rollout from a fixed start state
state, _ = cartpole_env.reset(seed=1)
real_states, model_states = [state], [state]
sim_state = state
for _ in range(15):
    action = cartpole_env.action_space.sample()
    real_next, _, terminated, truncated, _ = cartpole_env.step(action)
    real_states.append(real_next)
    sim_state, _ = forward_model.predict(sim_state, action)
    model_states.append(sim_state)
    if terminated or truncated:
        break

real_states = np.array(real_states)
model_states = np.array(model_states)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(real_states[:, 2], label='real (pole angle)', marker='o')
ax.plot(model_states[:, 2], label='model (pole angle)', marker='x')
ax.set_xlabel('Step')
ax.set_ylabel('Pole angle (rad)')
ax.set_title('Learned forward model vs. real environment rollout')
ax.legend()
plt.tight_layout()
plt.show()

## Part 2: Dyna-Q - Integrating Learning and Planning

Dyna-Q keeps a tabular Q-function and a tabular model. Every real step updates both Q and the model; between real steps, extra Q-updates are drawn from **simulated** experience replayed through the model. This squeezes more learning out of every real environment interaction.

In [ ]:
class GridWorld:
    """Deterministic grid MDP: -1 per step, +10 at the goal."""

    def __init__(self, size=6, goal=(5, 5), obstacles=None):
        self.size = size
        self.goal = goal
        self.obstacles = set(obstacles) if obstacles else {(2, 2), (2, 3), (3, 2)}
        self.actions = [0, 1, 2, 3]  # up, down, left, right
        self.deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def reset(self):
        self.pos = (0, 0)
        return self.pos

    def step(self, action):
        di, dj = self.deltas[action]
        ni, nj = self.pos[0] + di, self.pos[1] + dj
        if not (0 <= ni < self.size and 0 <= nj < self.size) or (ni, nj) in self.obstacles:
            ni, nj = self.pos
        self.pos = (ni, nj)
        done = self.pos == self.goal
        reward = 10.0 if done else -1.0
        return self.pos, reward, done


class DynaQAgent:
    """Tabular Q-learning with a learned deterministic model and background planning."""

    def __init__(self, actions, alpha=0.3, gamma=0.95, epsilon=0.1, planning_steps=10):
        self.actions = actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.planning_steps = planning_steps
        self.Q = defaultdict(lambda: {a: 0.0 for a in actions})
        self.model = {}  # (s, a) -> (s', r)
        self.seen = []   # list of (s, a) visited, for sampling during planning

    def act(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.choice(self.actions)
        qs = self.Q[state]
        return max(qs, key=qs.get)

    def _q_update(self, s, a, r, s_next):
        best_next = max(self.Q[s_next].values())
        self.Q[s][a] += self.alpha * (r + self.gamma * best_next - self.Q[s][a])

    def update(self, s, a, r, s_next):
        self._q_update(s, a, r, s_next)
        if (s, a) not in self.model:
            self.seen.append((s, a))
        self.model[(s, a)] = (s_next, r)
        for _ in range(self.planning_steps):
            ps, pa = self.seen[np.random.randint(len(self.seen))]
            ps_next, pr = self.model[(ps, pa)]
            self._q_update(ps, pa, pr, ps_next)


def run_dyna_q(planning_steps, n_episodes=40, max_steps=200):
    env = GridWorld()
    agent = DynaQAgent(env.actions, planning_steps=planning_steps)
    steps_per_episode = []
    for _ in range(n_episodes):
        state = env.reset()
        for step in range(max_steps):
            action = agent.act(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state)
            state = next_state
            if done:
                break
        steps_per_episode.append(step + 1)
    return steps_per_episode


steps_dyna = run_dyna_q(planning_steps=10)
steps_qlearning = run_dyna_q(planning_steps=0)  # planning_steps=0 recovers plain Q-learning

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps_qlearning, label='Q-learning (0 planning steps)')
ax.plot(steps_dyna, label='Dyna-Q (10 planning steps)')
ax.set_xlabel('Episode')
ax.set_ylabel('Steps to reach goal')
ax.set_title('Dyna-Q reaches a good policy in far fewer real episodes')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Episodes to first reach <= 15 steps - Q-learning: "
      f"{next((i for i, s in enumerate(steps_qlearning) if s <= 15), 'not reached')}, "
      f"Dyna-Q: {next((i for i, s in enumerate(steps_dyna) if s <= 15), 'not reached')}")

## Part 3: Model Predictive Control with the Learned Forward Model

MPC re-plans at every step: sample candidate action sequences, roll them forward through the **learned** forward model, score them by predicted return, execute only the first action of the best sequence, then replan. This is random-shooting MPC — the simplest member of the CEM/MPC family.

CartPole hands out a constant `+1` reward per step regardless of how the pole is tilted, even on the terminating step — so the reward alone carries no planning signal. What actually matters is *how long the episode survives*. We give the planner that signal directly: apply CartPole's own (documented, publicly known) termination thresholds to the model's predicted states and stop accumulating reward once a rollout predicts the pole has fallen or the cart has left the track. This mirrors a common MBRL setup where the reward/termination function is known and only the dynamics are learned.

In [ ]:
CARTPOLE_X_THRESHOLD = 2.4
CARTPOLE_THETA_THRESHOLD = 12 * 2 * math.pi / 360  # radians


def is_terminal(state):
    x, _, theta, _ = state
    return abs(x) > CARTPOLE_X_THRESHOLD or abs(theta) > CARTPOLE_THETA_THRESHOLD


def rollout_return(model, state, action_sequence, gamma=0.99):
    total, discount = 0.0, 1.0
    s = state
    for a in action_sequence:
        if is_terminal(s):
            break
        s, r = model.predict(s, a)
        total += discount * r
        discount *= gamma
    return total


def mpc_action(model, state, n_actions, horizon=8, n_candidates=64):
    best_return, best_action = -np.inf, 0
    for _ in range(n_candidates):
        sequence = np.random.randint(0, n_actions, size=horizon)
        ret = rollout_return(model, state, sequence)
        if ret > best_return:
            best_return, best_action = ret, sequence[0]
    return best_action


def evaluate_policy(action_fn, n_episodes=10, max_steps=200):
    env = gym.make('CartPole-v1')
    returns = []
    for ep in range(n_episodes):
        state, _ = env.reset(seed=100 + ep)
        total = 0.0
        for _ in range(max_steps):
            action = action_fn(state)
            state, reward, terminated, truncated, _ = env.step(action)
            total += reward
            if terminated or truncated:
                break
        returns.append(total)
    env.close()
    return returns


random_returns = evaluate_policy(lambda s: np.random.randint(n_actions))
mpc_returns = evaluate_policy(lambda s: mpc_action(forward_model, s, n_actions))

print(f"Random policy   - mean return: {np.mean(random_returns):.1f}")
print(f"MPC (forward model) - mean return: {np.mean(mpc_returns):.1f}")

## Part 4: Monte Carlo Tree Search (MCTS)

MCTS builds a search tree online: **select** via UCB1, **expand** an untried action, **simulate** a random rollout to estimate value, then **backpropagate** the return up the tree. Unlike the MPC above, this uses the true simulator (via `deepcopy`) rather than a learned model — the classical MCTS setting.

In [ ]:
class MCTSNode:
    def __init__(self, n_actions, parent=None, action=None):
        self.n_actions = n_actions
        self.parent = parent
        self.action = action
        self.children = {}
        self.visits = 0
        self.value_sum = 0.0
        self.untried_actions = list(range(n_actions))

    def value(self):
        return self.value_sum / self.visits if self.visits > 0 else 0.0

    def ucb1(self, c=1.4):
        if self.visits == 0:
            return np.inf
        return self.value() + c * math.sqrt(math.log(self.parent.visits) / self.visits)

    def best_child(self):
        return max(self.children.values(), key=lambda child: child.ucb1())


def rollout_policy(env, max_steps=50, gamma=0.99):
    total, discount = 0.0, 1.0
    for _ in range(max_steps):
        _, reward, terminated, truncated, _ = env.step(env.action_space.sample())
        total += discount * reward
        discount *= gamma
        if terminated or truncated:
            break
    return total


def mcts_search(root_env, n_actions, n_simulations=40):
    root = MCTSNode(n_actions)
    root_env_state = copy.deepcopy(root_env)

    for _ in range(n_simulations):
        node = root
        env = copy.deepcopy(root_env_state)
        done = False

        # Selection
        while not node.untried_actions and node.children and not done:
            node = node.best_child()
            _, _, terminated, truncated, _ = env.step(node.action)
            done = terminated or truncated

        # Expansion
        if node.untried_actions and not done:
            action = node.untried_actions.pop(np.random.randint(len(node.untried_actions)))
            _, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            child = MCTSNode(n_actions, parent=node, action=action)
            node.children[action] = child
            node = child

        # Simulation
        value = 0.0 if done else rollout_policy(env)

        # Backpropagation
        while node is not None:
            node.visits += 1
            node.value_sum += value
            node = node.parent

    return max(root.children.items(), key=lambda kv: kv[1].visits)[0]


def mcts_action_fn(state_env):
    return lambda state: mcts_search(state_env, n_actions, n_simulations=40)


mcts_env = gym.make('CartPole-v1')
mcts_returns = []
for ep in range(5):
    state, _ = mcts_env.reset(seed=200 + ep)
    total = 0.0
    for _ in range(200):
        action = mcts_search(mcts_env, n_actions, n_simulations=40)
        state, reward, terminated, truncated, _ = mcts_env.step(action)
        total += reward
        if terminated or truncated:
            break
    mcts_returns.append(total)
mcts_env.close()

print(f"MCTS (true simulator, 40 sims/step) - mean return over 5 episodes: {np.mean(mcts_returns):.1f}")

## Part 5: Sketch - AlphaZero-Style Search

AlphaZero replaces MCTS's random rollout with a **policy-value network**: the policy head biases which actions to expand (via PUCT instead of UCB1), and the value head replaces the rollout as the leaf estimate. A full implementation trains this network from self-play data over many iterations — out of scope here. This sketch shows the search-side change: swap the rollout-based leaf evaluation for a network evaluation, and bias selection with learned priors.

In [ ]:
class PolicyValueNet(nn.Module):
    """Sketch of an AlphaZero-style network: action priors + a state value."""

    def __init__(self, state_dim, n_actions, hidden=64):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(state_dim, hidden), nn.ReLU())
        self.policy_head = nn.Linear(hidden, n_actions)
        self.value_head = nn.Linear(hidden, 1)

    def forward(self, state):
        h = self.trunk(state)
        return torch.softmax(self.policy_head(h), dim=-1), torch.tanh(self.value_head(h))

    def evaluate(self, state):
        with torch.no_grad():
            s = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
            probs, value = self.forward(s)
        return probs.squeeze(0).cpu().numpy(), value.item()


class PUCTNode(MCTSNode):
    def __init__(self, n_actions, prior, parent=None, action=None):
        super().__init__(n_actions, parent, action)
        self.prior = prior

    def puct(self, c=1.5):
        exploit = self.value()
        explore = c * self.prior * math.sqrt(self.parent.visits) / (1 + self.visits)
        return exploit + explore


def puct_search(root_env, net, n_actions, n_simulations=20):
    """Same select/expand/backprop skeleton as mcts_search, but the leaf value comes
    from net.evaluate() instead of a random rollout, and expansion is guided by priors."""
    root_env_state = copy.deepcopy(root_env)
    root_probs, _ = net.evaluate(root_env_state.unwrapped.state)
    root = PUCTNode(n_actions, prior=1.0)

    for _ in range(n_simulations):
        node = root
        env = copy.deepcopy(root_env_state)
        done = False

        while not node.untried_actions and node.children and not done:
            node = max(node.children.values(), key=lambda c: c.puct())
            _, _, terminated, truncated, _ = env.step(node.action)
            done = terminated or truncated

        value = 0.0
        if node.untried_actions and not done:
            action = node.untried_actions.pop(0)
            _, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            probs, value = net.evaluate(env.unwrapped.state) if not done else ([1.0] * n_actions, 0.0)
            child = PUCTNode(n_actions, prior=root_probs[action], parent=node, action=action)
            node.children[action] = child
            node = child

        while node is not None:
            node.visits += 1
            node.value_sum += value
            node = node.parent

    return max(root.children.items(), key=lambda kv: kv[1].visits)[0] if root.children else 0


sketch_net = PolicyValueNet(state_dim, n_actions).to(device)
sketch_env = gym.make('CartPole-v1')
state, _ = sketch_env.reset(seed=42)
action = puct_search(sketch_env, sketch_net, n_actions, n_simulations=20)
print(f"PUCT search (untrained network) picked action: {action}")
print("A full AlphaZero loop would alternate self-play (generate data with puct_search)")
print("and supervised training of PolicyValueNet on (state -> visit-count policy, outcome).")

## Part 6: Stable-Baselines3 Reference

SB3 has no built-in Dyna-Q or MCTS agent, so we use SB3's **DQN** as the model-free baseline: it validates that our CartPole setup is solvable, and gives a real-environment-step count to compare against the model-based methods above. DQN is purely model-free, so it needs many more real environment steps (30,000) than our forward model needed transitions (4,000) to reach comparable performance — the sample-efficiency case for model-based RL made concrete.

In [ ]:
from stable_baselines3 import DQN

sb3_env = gym.make('CartPole-v1')
sb3_model = DQN('MlpPolicy', sb3_env, learning_rate=1e-3, buffer_size=20000, verbose=0,
                 exploration_fraction=0.3, target_update_interval=500)
sb3_model.learn(total_timesteps=30000)

sb3_returns = []
for ep in range(10):
    state, _ = sb3_env.reset(seed=300 + ep)
    total = 0.0
    for _ in range(200):
        action, _ = sb3_model.predict(state, deterministic=True)
        state, reward, terminated, truncated, _ = sb3_env.step(action)
        total += reward
        if terminated or truncated:
            break
    sb3_returns.append(total)
sb3_env.close()

print(f"SB3 DQN (30,000 real env steps) - mean return over 10 episodes: {np.mean(sb3_returns):.1f}")
print(f"MPC (4,000 transitions to fit the model) - mean return: {np.mean(mpc_returns):.1f}")

## Key Takeaways

1. **Forward models** turn dynamics learning into supervised regression on collected transitions
2. **Dyna-Q** reuses every real transition many times via background planning — fewer real episodes for the same policy quality
3. **MPC** (random-shooting) plans directly against a learned model at every step, no explicit policy needed
4. **MCTS** (select/expand/simulate/backpropagate with UCB1) searches online using the true simulator
5. **AlphaZero** replaces MCTS's random rollout with a trained value estimate and biases selection with a trained policy prior (PUCT) — full training requires an iterative self-play loop, sketched but not run here
6. Model error compounds with planning horizon — short horizons and frequent replanning (as in MPC) keep this in check